<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 7


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс BankAccount в C#, который будет представлять 
информацию об учетных записях в банке. На основе этого класса разработать 2-3 
производных класса, демонстрирующих принципы наследования и полиморфизма. 
В каждом из классов должны быть реализованы новые атрибуты и методы, а также 
переопределены некоторые методы базового класса для демонстрации 
полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;

public delegate void AccountEventHandler(BankAccount account, string message);

public interface IAccountActions
{
    void Deposit(decimal amount);
    void Withdraw(decimal amount);
    string GetInfo();
}

public abstract class BankAccount : IAccountActions
{
    public string AccountNumber { get; set; }
    public decimal Balance { get; set; }
    public string AccountType { get; protected set; }
    public string Owner { get; set; }

    public DateTime CreatedAt { get; set; }
    public bool IsActive { get; set; }
    public string Currency { get; set; }

    public BankAccount(string accNum, decimal balance, string type, string owner)
    {
        AccountNumber = accNum;
        Balance = balance;
        AccountType = type;
        Owner = owner;
        CreatedAt = DateTime.Now;
        IsActive = true;
        Currency = "RUB";
    }

    public abstract void Deposit(decimal amount);
    public abstract void Withdraw(decimal amount);
    public abstract string GetInfo();

    public void Deactivate() => IsActive = false;
    public void ChangeCurrency(string newCurrency) => Currency = newCurrency;
    public void PrintOwner() => Console.WriteLine(Owner);
}

public class SavingsAccount : BankAccount, IAccountActions
{
    public decimal InterestRate { get; set; }
    public int SavingRank { get; set; }
    public bool AutoReplenish { get; set; }

    public SavingsAccount(string accNum, decimal balance, string owner, decimal interest)
        : base(accNum, balance, "Сберегательный", owner)
    {
        InterestRate = interest;
        SavingRank = 1;
        AutoReplenish = true;
    }

    void IAccountActions.Deposit(decimal amount)
    {
        Balance += amount + amount * InterestRate;
    }

    void IAccountActions.Withdraw(decimal amount)
    {
        if (amount <= Balance) Balance -= amount;
    }

    string IAccountActions.GetInfo()
    {
        return $"{AccountNumber,-10} | {Balance,12:C} | {AccountType,-15} | {Owner,-10} | %: {InterestRate:P} | Rank {SavingRank}";
    }

    public override void Deposit(decimal amount) => ((IAccountActions)this).Deposit(amount);
    public override void Withdraw(decimal amount) => ((IAccountActions)this).Withdraw(amount);
    public override string GetInfo() => ((IAccountActions)this).GetInfo();
}

public class CheckingAccount : BankAccount, IAccountActions
{
    public decimal OverdraftLimit { get; set; }
    public bool SmsNotification { get; set; }
    public string BankBranch { get; set; }

    public CheckingAccount(string accNum, decimal balance, string owner, decimal overdraft)
        : base(accNum, balance, "Текущий", owner)
    {
        OverdraftLimit = overdraft;
        SmsNotification = true;
        BankBranch = "Main Branch";
    }

    void IAccountActions.Deposit(decimal amount)
    {
        if (amount > 0) Balance += amount;
    }

    void IAccountActions.Withdraw(decimal amount)
    {
        if (amount <= Balance + OverdraftLimit) Balance -= amount;
    }

    string IAccountActions.GetInfo()
    {
        return $"{AccountNumber,-10} | {Balance,12:C} | {AccountType,-15} | {Owner,-10} | Овердрафт: {OverdraftLimit:C}";
    }

    public override void Deposit(decimal amount) => ((IAccountActions)this).Deposit(amount);
    public override void Withdraw(decimal amount) => ((IAccountActions)this).Withdraw(amount);
    public override string GetInfo() => ((IAccountActions)this).GetInfo();
}

public class InvestmentAccount : BankAccount, IAccountActions
{
    public List<string> Assets { get; set; }
    public decimal PortfolioRisk { get; set; }
    public bool AutoRebalance { get; set; }

    public InvestmentAccount(string accNum, decimal balance, string owner, List<string> assets)
        : base(accNum, balance, "Инвестиционный", owner)
    {
        Assets = assets;
        PortfolioRisk = 0.15m;
        AutoRebalance = true;
    }

    void IAccountActions.Deposit(decimal amount)
    {
        if (amount > 0) Balance += amount;
    }

    void IAccountActions.Withdraw(decimal amount)
    {
        if (amount <= Balance) Balance -= amount;
    }

    public void AddAsset(string asset)
    {
        Assets.Add(asset);
    }

    public void RemoveAsset(string asset)
    {
        Assets.Remove(asset);
    }

    string IAccountActions.GetInfo()
    {
        return $"{AccountNumber,-10} | {Balance,12:C} | {AccountType,-15} | {Owner,-10} | Активы: {string.Join(", ", Assets)}";
    }

    public override void Deposit(decimal amount) => ((IAccountActions)this).Deposit(amount);
    public override void Withdraw(decimal amount) => ((IAccountActions)this).Withdraw(amount);
    public override string GetInfo() => ((IAccountActions)this).GetInfo();
}

public class AccountManager<T> where T : BankAccount
{
    public event AccountEventHandler AccountAdded;
    public event AccountEventHandler AccountChanged;

    private List<T> accounts = new List<T>();
    private Dictionary<string, T> accountsDict = new Dictionary<string, T>();
    private HashSet<string> owners = new HashSet<string>();

    public void AddAccount(T account)
    {
        accounts.Add(account);
        accountsDict[account.AccountNumber] = account;
        owners.Add(account.Owner);
        AccountAdded?.Invoke(account, "Счет добавлен");
    }

    public void ModifyAccount(T account, decimal deposit)
    {
        account.Deposit(deposit);
        AccountChanged?.Invoke(account, "Баланс изменён");
    }

    public void PrintAllAccounts()
    {
        Console.WriteLine(new string('-', 100));
        Console.WriteLine($"{"Номер",-10} | {"Баланс",12} | {"Тип",-15} | {"Владелец",-10} | Инфо");
        Console.WriteLine(new string('-', 100));
        foreach (var acc in accounts)
            Console.WriteLine(acc.GetInfo());
        Console.WriteLine(new string('-', 100));
    }
}

var manager = new AccountManager<BankAccount>();

manager.AccountAdded += (acc, msg) =>
{
    Console.WriteLine($"{msg}: {acc.AccountNumber} {acc.Owner}");
};

manager.AccountChanged += (acc, msg) =>
{
    Console.WriteLine($"{msg}: {acc.AccountNumber} {acc.Balance:C}");
};

var savings = new SavingsAccount("SA001", 1000, "Иван", 0.05m);
var checking = new CheckingAccount("CH001", 500, "Петр", 200);
var investment = new InvestmentAccount("IN001", 10000, "Мария", new List<string> { "Акции", "Облигации" });

manager.AddAccount(savings);
manager.AddAccount(checking);
manager.AddAccount(investment);

manager.ModifyAccount(savings, 300);
manager.ModifyAccount(investment, 1500);

investment.AddAsset("ETF");

Console.WriteLine();
manager.PrintAllAccounts();


Счет добавлен: SA001 Иван
Счет добавлен: CH001 Петр
Счет добавлен: IN001 Мария
Баланс изменён: SA001 ¤1,315.00
Баланс изменён: IN001 ¤11,500.00

----------------------------------------------------------------------------------------------------
Номер      |       Баланс | Тип             | Владелец   | Инфо
----------------------------------------------------------------------------------------------------
SA001      |    ¤1,315.00 | Сберегательный  | Иван       | %: 5.000% | Rank 1
CH001      |      ¤500.00 | Текущий         | Петр       | Овердрафт: ¤200.00
IN001      |   ¤11,500.00 | Инвестиционный  | Мария      | Активы: Акции, Облигации, ETF
----------------------------------------------------------------------------------------------------
